# 01 - BBOB Problem Noise Landscape Analysis

This notebook performs comprehensive **BBOB landscape analysis**, **additive Gaussian noise injection**, **topology visualization**, and **empirical noise distribution analysis** across multiple noise standard deviations (`noise_level = [0.0, 0.05, 0.2, 0.5, 0.8, 1.0, 2.0, 5.0]`) using interactive Plotly graphics.

In [43]:
# Setup paths and imports
import sys
from pathlib import Path
# Add project root src/ to sys.path regardless of directory name or launch folder
root_dir = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
src_path = str(root_dir / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)


import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import rankdata
from core.problems.bbob import BBOBProblem


## 1. Unified Experiment Configuration
Define problem parameters, noise standard deviation levels, and a global color palette used consistently across all analysis cells.

In [44]:
problem_id = 11
dim = 2
instance_id = 1

# List of standard deviation noise levels to analyze
noisy_levels = [0.05, 0.1, 0.2]
all_levels = [0.0] + noisy_levels

# Global color palette mapping noise standard deviations to colors
color_palette = {
    0.05: '#10B981',  # emerald green
    0.2:  '#3B82F6',  # blue
    0.5:  '#8B5CF6',  # purple
    0.8:  '#F59E0B',  # amber
    1.0:  '#EC4899',  # pink
    2.0:  '#EF4444',  # red
    5.0:  '#6B7280'   # gray
}

problem = BBOBProblem(problem_id=problem_id, dim=dim, instance_id=instance_id)

print(f"Loaded BBOB Problem {problem_id} ({dim}D)")
print(f"Domain Bounds: [{problem.lower_bound}, {problem.upper_bound}] (vector lb={problem.lb}, ub={problem.ub})")
print(f"True target optimum (global minimum): {problem.true_optimum:.4f}")
print(f"Configured Noise Levels (std): {all_levels}")


Loaded BBOB Problem 11 (2D)
Domain Bounds: [-5.0, 5.0] (vector lb=[-5. -5.], ub=[5. 5.])
True target optimum (global minimum): 76.2700
Configured Noise Levels (std): [0.0, 0.05, 0.1, 0.2]


## 2. 1D Sample Point Fitness Comparison
Evaluate 100 sample points across the search space and compare clean fitness against noisy fitness for each configured `noise_level` level.

In [45]:
np.random.seed(42)
lb, ub = problem.lower_bound, problem.upper_bound
points = np.random.uniform(lb, ub, (100, dim))

clean_y = np.array([problem(x) for x in points])
sort_idx = np.argsort(clean_y)
sorted_clean = clean_y[sort_idx]
indices = np.arange(len(points))

# Combined Plot showing all noise std levels overlaid on clean ground truth
fig_combined = go.Figure()
fig_combined.add_trace(go.Scatter(
    x=indices,
    y=sorted_clean,
    mode='lines',
    name='Clean Fitness (Ground Truth)',
    line=dict(color='#111827', width=3.5)
))

# Add true optimum line
fig_combined.add_hline(
    y=problem.true_optimum,
    line_dash="dash",
    line_color="red",
    annotation_text="True Optimum",
    annotation_position="bottom right"
)

for level in noisy_levels:
    noisy_y = np.array([problem.add_noise(y, level) for y in clean_y])
    sorted_noisy = noisy_y[sort_idx]
    fig_combined.add_trace(go.Scatter(
        x=indices,
        y=sorted_noisy,
        mode='markers',
        name=f'level = {level}',
        marker=dict(color=color_palette.get(level, '#6B7280'), size=5, opacity=0.75)
    ))

fig_combined.update_layout(
    title=f"BBOB Problem {problem_id}: Clean vs. Noisy Fitness across Noise Levels {noisy_levels}",
    xaxis_title="Sample Points (Sorted by Clean Fitness)",
    yaxis_title="Fitness Value",
    template="plotly_white",
    hovermode='x unified',
    height=600,
    width=1000
)
fig_combined.show()

# Subplot Grid for side-by-side noise inspection dynamically sized
n_plots = len(noisy_levels)
cols = 4
rows = int(np.ceil(n_plots / cols))
fig_subplots = make_subplots(
    rows=rows, cols=cols,
    subplot_titles=[f'noise_level = {level}' for level in noisy_levels]
)

coords = [(r + 1, c + 1) for r in range(rows) for c in range(cols)][:n_plots]
for idx, (level, (r, c)) in enumerate(zip(noisy_levels, coords), start=1):
    noisy_y = np.array([problem.add_noise(y, level) for y in clean_y])
    sorted_noisy = noisy_y[sort_idx]
    
    # Clean reference line
    fig_subplots.add_trace(
        go.Scatter(x=indices, y=sorted_clean, mode='lines', name='Clean', line=dict(color='#111827', width=2), showlegend=(idx==1)),
        row=r, col=c
    )
    # Noisy markers
    fig_subplots.add_trace(
        go.Scatter(x=indices, y=sorted_noisy, mode='markers', name=f'level={level}', marker=dict(color=color_palette.get(level, '#EF4444'), size=5, opacity=0.75), showlegend=(idx==1)),
        row=r, col=c
    )
    # Add true optimum line to subplot
    fig_subplots.add_hline(
        y=problem.true_optimum,
        line_dash="dash",
        line_color="red",
        row=r, col=c
    )

fig_subplots.update_layout(
    title_text=f"BBOB Problem {problem_id}: Side-by-Side Noise Standard Deviation Grid",
    height=max(350, 300 * rows),
    width=1150,
    template="plotly_white"
)
fig_subplots.show()


## 3. 2D Contour Heatmap Comparison
Generate 2D normalized rank contour maps across all standard deviation noise levels to visualize how noise distorts landscape contours and global minimum accessibility.

In [46]:
from scipy.stats import rankdata

grid_resol = 100
x_range = np.linspace(problem.lower_bound, problem.upper_bound, grid_resol)
y_range = np.linspace(problem.lower_bound, problem.upper_bound, grid_resol)
X, Y = np.meshgrid(x_range, y_range)

landscapes = {}
norm_landscapes = {}

for level in all_levels:
    # Compute grid landscape values
    Z = np.zeros_like(X)
    for i in range(grid_resol):
        for j in range(grid_resol):
            pt = np.copy(problem.optimum_x)
            pt[0] = X[i, j]
            pt[1] = Y[i, j]
            res = problem(pt, noise_level=level) if level > 0.0 else problem(pt)
            Z[i, j] = res.get(level, float('inf')) if isinstance(res, dict) else float(res)
            
    landscapes[level] = Z
    
    # Calculate normalized rank for heatmap visualization
    Z_flat = Z.flatten()
    Z_rank = rankdata(Z_flat)
    Z_norm = (Z_rank - Z_rank.min()) / (Z_rank.max() - Z_rank.min())
    norm_landscapes[level] = Z_norm.reshape(Z.shape)

n_plots = len(all_levels)
cols = 4
rows = int(np.ceil(n_plots / cols))
fig_contour = make_subplots(
    rows=rows, cols=cols,
    subplot_titles=[f'noise_level = {level} (Clean)' if level == 0.0 else f'noise_level = {level}' for level in all_levels]
)

coords_contour = [(r + 1, c + 1) for r in range(rows) for c in range(cols)][:n_plots]
opt_x, opt_y = problem.optimum_x[:2]

for level, (r, c) in zip(all_levels, coords_contour):
    # Contour heatmap using Normalized Rank
    fig_contour.add_trace(
        go.Contour(
            x=x_range, 
            y=y_range, 
            z=norm_landscapes[level], 
            colorscale='Viridis', 
            showscale=(r==1 and c==4),
            contours=dict(coloring='heatmap', showlines=True)
        ),
        row=r, col=c
    )
    # Add white 'x' at the true global optimum
    fig_contour.add_trace(
        go.Scatter(
            x=[opt_x], y=[opt_y], 
            mode='markers', 
            marker=dict(color='white', symbol='x', size=8), 
            showlegend=False,
            name='Global Minimum'
        ),
        row=r, col=c
    )

fig_contour.update_layout(
    title_text=f"Normalized Rank Heatmap Comparison across Noise Levels<br><sup>BBOB Problem {problem_id} | f_opt = {problem.true_optimum:.4f}</sup>", 
    height=max(400, 350 * rows), 
    width=1350
)
fig_contour.show()


## 4. 3D Surface Topography Comparison
Render 3D surface plots for all noise standard deviation levels (`all_levels`) to analyze surface roughness and gradient degradation.

In [47]:
fig_3d_titles = [f"noise_level = {level} ({'Clean' if level==0.0 else 'Noisy'})" for level in all_levels]
n_plots = len(all_levels)
cols = 4
rows = int(np.ceil(n_plots / cols))
specs = [[{'type': 'surface'} for _ in range(cols)] for _ in range(rows)]
fig_3d = make_subplots(
    rows=rows, cols=cols,
    specs=specs,
    subplot_titles=fig_3d_titles,
    horizontal_spacing=0.08,
    vertical_spacing=0.15
)

coords_3d = [(r + 1, c + 1) for r in range(rows) for c in range(cols)][:n_plots]
colorscales_3d = ['Viridis', 'Cividis', 'Blues', 'Purples', 'Oranges', 'Plasma', 'Inferno', 'Magma']
if len(colorscales_3d) < n_plots:
    colorscales_3d = (colorscales_3d * int(np.ceil(n_plots / len(colorscales_3d))))[:n_plots]

opt_x, opt_y = problem.optimum_x[:2]

for level, (r, c), cscale in zip(all_levels, coords_3d, colorscales_3d):
    fig_3d.add_trace(
        go.Surface(
            x=x_range, y=y_range, 
            z=landscapes[level], 
            colorscale=cscale, 
            showscale=False
        ),
        row=r, col=c
    )
    # Add marker at the global optimum coordinates in 3D
    fig_3d.add_trace(
        go.Scatter3d(
            x=[opt_x], y=[opt_y], z=[problem.true_optimum],
            mode='markers',
            marker=dict(color='red', size=6, symbol='diamond', line=dict(color='white', width=1)),
            name='Global Optimum',
            showlegend=False
        ),
        row=r, col=c
    )

fig_3d.update_scenes(
    xaxis_title='x₁',
    yaxis_title='x₂',
    zaxis_title='f(x)',
    aspectmode='cube',
    camera=dict(eye=dict(x=1.7, y=1.7, z=1.2))
)

fig_3d.update_layout(
    title_text=f"3D Surface Topography Comparison<br><sup>BBOB Problem {problem_id} | bounds = [{problem.lower_bound}, {problem.upper_bound}]</sup>",
    height=max(500, 450 * rows),
    width=1600,
    margin=dict(l=40, r=40, t=110, b=40)
)
fig_3d.show()


## 5. Empirical Noise Distribution Analysis
Evaluate 1000 repeated noisy samples at the origin `(0, 0)` for each noisy standard deviation level to confirm that empirical noise matches Gaussian statistics.

In [48]:
n_samples = 1000
test_pt = np.zeros(dim)

fig_hist = go.Figure()
for level in noisy_levels:
    samples = [problem(test_pt, noise_level=level)[level] for _ in range(n_samples)]
    fig_hist.add_trace(go.Histogram(
        x=samples,
        opacity=0.45,
        name=f'level={level} (emp std={np.std(samples):.2f})',
        marker_color=color_palette.get(level, '#6B7280'),
        nbinsx=30
    ))

true_val = problem(test_pt)
fig_hist.add_vline(x=true_val, line_dash="dash", line_color="black", annotation_text="True Value")

fig_hist.update_layout(
    title_text="Empirical Noise Distribution across 1000 Evaluations at Origin for all Noise Levels",
    xaxis_title="Evaluated Fitness Value",
    yaxis_title="Frequency",
    barmode='overlay',
    height=500,
    width=1000
)
fig_hist.show()
